# Anvil Quick Start Notebook

This notebook walks you through your first tasks with Anvil, the execution engine of the FableForge ecosystem.

**What you'll learn:**
- Installing and configuring Anvil
- Running your first coding task
- Understanding verification output
- Handling errors with recovery
- Seeing the Plan→Execute→Verify→Recover loop in action

## 1. Installation

In [ ]:
# Install Anvil with all dependencies
!pip install anvil[all] -q

# Verify installation
import anvil
print(f"Anvil version: {anvil.__version__}")

# List available tools
from anvil.tools import ToolExecutor
executor = ToolExecutor()
for tool in executor.list_tools():
    print(f"  {tool.name:15s} {tool.description}")

## 2. First Task — Simple Code Generation

In [ ]:
from anvil.engine import AnvilEngine, AnvilConfig

# Create engine with local model (Ollama)
# Make sure Ollama is running: ollama serve
# And the model is pulled: ollama pull fableforge-14b
engine = AnvilEngine(
    model_backend="local",
    model_name="fableforge-14b",
    workspace=".",  # Current directory
)

print("Engine created successfully!")
print(f"Tools available: {len(engine.tools)}")
print(f"Verification enabled: {engine.verify}")

In [ ]:
# Run a simple task
result = engine.run(
    "Create a Python file called calculator.py with add, subtract, multiply, and divide functions."
)

# Display results using Rich
from rich.console import Console
from rich.panel import Panel
from rich.table import Table

console = Console()

console.print(Panel(
    f"[bold green]Task Complete![/bold green]\n"
    f"Duration: {result.duration_seconds:.1f}s\n"
    f"Iterations: {len(result.iterations)}\n"
    f"Success: {result.success}",
    title="Result Summary",
))

## 3. Understanding Verification Output

In [ ]:
# Dive into verification results
verify = result.verification

console.print(f"\n[bold]Verification Report[/bold]")
console.print(f"  Passed: {'✓' if verify.passed else '✗'}")
console.print(f"  Score:  {verify.score:.2f}/1.00")
console.print(f"  Time:   {verify.duration_ms:.0f}ms")

# Show individual checker results
table = Table(title="Checker Results")
table.add_column("Checker", style="cyan")
table.add_column("Passed", style="green")
table.add_column("Message")
table.add_column("Severity")

for check in verify.checkers:
    status = "✓" if check.passed else "✗"
    style = "green" if check.passed else "red"
    table.add_row(check.checker, f"[{style}]{status}[/{style}]", check.message, check.severity)

console.print(table)

## 4. Error Recovery in Action

In [ ]:
# Create a task that will initially fail (introducing a deliberate bug)
result = engine.run(
    "Add a function called divide that handles division by zero properly "
    "in calculator.py. The function should raise a ValueError with message "
    "'Cannot divide by zero' when the divisor is 0.",
    max_iterations=10,
)

console.print(Panel(
    f"[bold]Recovery Stats[/bold]\n"
    f"Total iterations: {len(result.iterations)}\n"
    f"Recovery attempts: {sum(1 for i in result.iterations if i.phase == 'recover')}\n"
    f"Final verification: {result.verification.score:.2f}",
    title="Recovery Results",
))

## 5. The Full Plan→Execute→Verify→Recover Loop

In [ ]:
import asyncio

# Stream the full loop to see each phase in real-time
console.print("[bold]Streaming Plan→Execute→Verify→Recover loop[/bold]\n")

async def stream_task():
    phases_seen = set()
    async for event in engine.run_stream(
        "Add a Fibonacci function to calculator.py with proper error handling "
        "for negative inputs and a comprehensive docstring"
    ):
        phase = event.phase
        if phase not in phases_seen:
            phases_seen.add(phase)
            console.print(f"\n[bold yellow]>>> {phase.upper()} PHASE[/bold yellow]")
        
        if phase == "plan":
            console.print(f"  [blue]Plan:[/blue] {event.content[:100]}")
        elif phase == "execute":
            console.print(f"  [green]Tool:[/green] {event.tool} — {event.content[:80]}")
        elif phase == "verify":
            status = "✓" if event.result else "✗"
            color = "green" if event.result else "red"
            console.print(f"  [{color}]{status}[/{color}] {event.check_name}")
        elif phase == "recover":
            console.print(f"  [yellow]Recovery:[/yellow] {event.content[:100]}")

asyncio.run(stream_task())
console.print("\n[bold green]Loop complete![/bold green]")

## 6. Session Continuity

In [ ]:
# Multi-turn conversation using sessions
result1 = engine.run(
    "Create a User class in models.py with name, email, and age fields",
)
session_id = result1.session_id
console.print(f"Session: {session_id}")
console.print(f"Result: {result1.summary}")

# Continue in the same session
result2 = engine.run(
    "Now add validation methods to the User class: validate_email() and validate_age()",
    session_id=session_id,
)
console.print(f"Continuation result: {result2.summary}")

# Add tests
result3 = engine.run(
    "Write unit tests for the User class and its validation methods",
    session_id=session_id,
)
console.print(f"Test result: {result3.summary}")
console.print(f"Verification: {result3.verification.score:.2f}")

## 7. Summary

You've learned:

1. **Installation** — `pip install anvil[all]`
2. **First task** — `engine.run(task)` with local model
3. **Verification** — Check results with `result.verification`
4. **Error recovery** — Automatic retry, rewrite, and escalation
5. **Streaming** — `engine.run_stream()` for real-time feedback
6. **Sessions** — Multi-turn conversations with `session_id`

Next steps:
- Try a different model backend (OpenAI, Anthropic)
- Explore custom verification checkers
- Set up the daemon for HTTP access
- Integrate with other FableForge projects